# document extractor

In [2]:
# https://onlyoneaman.medium.com/i-tested-7-python-pdf-extractors-so-you-dont-have-to-2025-edition-c88013922257

import textract
text = textract.process(r"/Users/karthickthangadurai/PersonalPro/My_Projects/Otelio/data/hotel_rag_document_v2.pdf").decode()

In [3]:
print(text)

Grand Azure Bay Hotel - Detailed Information
Guide
Overview
Grand Azure Bay Hotel is a luxury beachfront property offering premium accommodation, fine
dining, and personalized guest experiences. The hotel is designed to cater to both business and
leisure travelers with a focus on comfort, safety, and service excellence.

Location & Accessibility
The hotel is located in Elysian Coast City, approximately 5 km from the city center and 20 km from
the international airport. It is easily accessible via taxi, public transport, and hotel-arranged shuttle
services. Nearby attractions include beaches, cultural landmarks, shopping districts, and nightlife
hubs.

Frequently Asked Questions - General
Check-in time is 2:00 PM and check-out time is 11:00 AM. Early check-in and late check-out are
subject to availability. The hotel provides free Wi-Fi, complimentary breakfast for select bookings,
and 24/7 customer support.

Hygiene & Cleanliness Protocols
The hotel follows strict hygiene standards ali

In [4]:
# pip install "unstructured[all-docs]"
from unstructured.partition.auto import partition
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parent.parent if '__file__' in locals() else Path('.').resolve().parent

"""
__file__: A built-in Python variable that contains the file path of the current script (e.g., "/Users/karthick/projects/Otelio/src/config.py").

Path(__file__): Converts that path string into a robust pathlib.Path object, allowing you to use object-oriented path manipulations.

"""

print(f"PROJECT_ROOT: {PROJECT_ROOT}")

pdf_path = PROJECT_ROOT / "data" / "hotel_rag_document_v2.pdf"

blocks = partition(filename=pdf_path, strategy="fast")

for block in blocks:
    print(f"{block.category}: {block.text}")

PROJECT_ROOT: /Users/karthickthangadurai/PersonalPro/My_Projects/Otelio
Title: Grand Azure Bay Hotel - Detailed Information Guide
Title: Overview
NarrativeText: Grand Azure Bay Hotel is a luxury beachfront property offering premium accommodation, fine dining, and personalized guest experiences. The hotel is designed to cater to both business and leisure travelers with a focus on comfort, safety, and service excellence.
Title: Location & Accessibility
NarrativeText: The hotel is located in Elysian Coast City, approximately 5 km from the city center and 20 km from the international airport. It is easily accessible via taxi, public transport, and hotel-arranged shuttle services. Nearby attractions include beaches, cultural landmarks, shopping districts, and nightlife hubs.
Title: Frequently Asked Questions - General
NarrativeText: Check-in time is 2:00 PM and check-out time is 11:00 AM. Early check-in and late check-out are subject to availability. The hotel provides free Wi-Fi, complimen

In [4]:
"""Evaluated PyMuPDF, textract, and unstructured; chose unstructured because its element-type 
classification (Title/NarrativeText) enables deterministic section-based chunking, whereas the plain-text extractors 
collapse headings into body text and would require brittle heuristics."""


from unstructured.partition.pdf import partition_pdf

blocks = partition_pdf(
    filename="data/hotel_rag_document_v2.pdf",
    strategy="fast"
)

In [5]:
for block in blocks:
    print(f"{block.category}: {block.text}")

Title: Grand Azure Bay Hotel - Detailed Information Guide
Title: Overview
NarrativeText: Grand Azure Bay Hotel is a luxury beachfront property offering premium accommodation, fine dining, and personalized guest experiences. The hotel is designed to cater to both business and leisure travelers with a focus on comfort, safety, and service excellence.
Title: Location & Accessibility
NarrativeText: The hotel is located in Elysian Coast City, approximately 5 km from the city center and 20 km from the international airport. It is easily accessible via taxi, public transport, and hotel-arranged shuttle services. Nearby attractions include beaches, cultural landmarks, shopping districts, and nightlife hubs.
Title: Frequently Asked Questions - General
NarrativeText: Check-in time is 2:00 PM and check-out time is 11:00 AM. Early check-in and late check-out are subject to availability. The hotel provides free Wi-Fi, complimentary breakfast for select bookings, and 24/7 customer support.
Title: Hy

In [8]:
import pymupdf

doc = pymupdf.open("data/hotel_rag_document_v2.pdf")
for page in doc:
    print(page.get_text())

Grand Azure Bay Hotel - Detailed Information
Guide
Overview
Grand Azure Bay Hotel is a luxury beachfront property offering premium accommodation, fine
dining, and personalized guest experiences. The hotel is designed to cater to both business and
leisure travelers with a focus on comfort, safety, and service excellence.
Location & Accessibility
The hotel is located in Elysian Coast City, approximately 5 km from the city center and 20 km from
the international airport. It is easily accessible via taxi, public transport, and hotel-arranged shuttle
services. Nearby attractions include beaches, cultural landmarks, shopping districts, and nightlife
hubs.
Frequently Asked Questions - General
Check-in time is 2:00 PM and check-out time is 11:00 AM. Early check-in and late check-out are
subject to availability. The hotel provides free Wi-Fi, complimentary breakfast for select bookings,
and 24/7 customer support.

Hygiene & Cleanliness Protocols
The hotel follows strict hygiene standards aligne

# Chunking

In [5]:
from unstructured.chunking.title import chunk_by_title

chunks = chunk_by_title(blocks, combine_text_under_n_chars=0, include_orig_elements = True)

for chunk in chunks:
    print(chunk)
    print("\n\n" + "-"*80)

Grand Azure Bay Hotel - Detailed Information Guide


--------------------------------------------------------------------------------
Overview

Grand Azure Bay Hotel is a luxury beachfront property offering premium accommodation, fine dining, and personalized guest experiences. The hotel is designed to cater to both business and leisure travelers with a focus on comfort, safety, and service excellence.


--------------------------------------------------------------------------------
Location & Accessibility

The hotel is located in Elysian Coast City, approximately 5 km from the city center and 20 km from the international airport. It is easily accessible via taxi, public transport, and hotel-arranged shuttle services. Nearby attractions include beaches, cultural landmarks, shopping districts, and nightlife hubs.


--------------------------------------------------------------------------------
Frequently Asked Questions - General

Check-in time is 2:00 PM and check-out time is 11:00 

In [10]:
"""
Chunks carry a hotel metadata field; single-property today, but the schema and retrieval filter 
extend to multi-property without re-ingestion redesign.
"""

print(chunks[2].text)

Location & Accessibility

The hotel is located in Elysian Coast City, approximately 5 km from the city center and 20 km from the international airport. It is easily accessible via taxi, public transport, and hotel-arranged shuttle services. Nearby attractions include beaches, cultural landmarks, shopping districts, and nightlife hubs.


In [11]:
print(chunks[2].metadata.to_dict())

{'file_directory': 'data', 'filename': 'hotel_rag_document_v2.pdf', 'filetype': 'application/pdf', 'languages': ['eng'], 'last_modified': '2026-07-18T20:06:46', 'page_number': 1, 'orig_elements': 'eJzdU01v2zAM/SuEDzulnuPYjtzbUOwwYCgGLLegCGiJjonKkiHJWYKi/32S02Ldx2WXHXYk3yP1+EjtnzLSNJIJB1bZLWStwG0r13WNXVeJTa0kdVSWnewrIftWZCvIRgqoMGDkP2XSWqfYYCC/xBovdg6Hgfg4hJgR1ToXohWx7gX6xioMEanbOi+3dRORybIJqX6/34q8WEFZF3n7sILXsKmuYblpc/GHeKHHROYvPtCYJvnCZ9JfJ5SUPUegZ00HxY5ksO6SCMsIL4jBkVJusIH0weHxoKycF19OZT6p/pUYLtNCxGnSLDGwNe9fYI3mOONx8WGfkTlmD0vWh8NoFfdMi8NlUTY3xfZmLXZlcVs0t1WTqqdYeTDz2JGLrHWSHOicHMw+2+tD8A4+SEnec8eawyWVverZcdDLoL8utBMoGllto1MNtqJuKiRRUF8UqiQhy3+1UCHy9Y+FbjbVNazX67z+Pb7S/6eFLhn3Fx/t7QXsBoJlEmAPOp0DKWADH/XFMxq4s1EU3MWbWEEcxNkzj5GjL1DD4wi9syOE2ENGBsgoghygUVAWP8GcALN4gBqQ3WRdyOFTSM8Seo4N8eUCNcGJEQKeeQXT3EXzIDg0PtWslu6L4ht0MXuMev0wh3ik4MmdODbJ4Z7QdbFliIUyveqjBKlnRdARyoH8CuSsw+yinLgNNaJ7jDk/2GlicwTFPjiWwV8fNOlENffRrLnz+dv/cZ9kBD7RLln6/PAdQzaR9Q=='}


# create schema for ChromaDB

In [6]:
from unstructured.staging.base import elements_from_base64_gzipped_json

records = []

hotel_id = str(chunks[0]).split('-')[0].strip().lower().replace(" ", "-")
hotel_name = str(chunks[0]).split('-')[0].strip()

for element, counter in zip(chunks[1:], range(1, len(chunks[1:]) + 1)):

    
    metadata = element.metadata.to_dict()

    id = f"{hotel_id}-{counter}"
    page_number = metadata["page_number"]
    title = ''
    
    content = ''

    orig_elements = elements_from_base64_gzipped_json(metadata["orig_elements"])

    for orig_element in orig_elements:
        
        content += orig_element.text + " : "

        if orig_element.category == "Title":

            title = orig_element.text

    doc = {
        'metadata': {
            'id': id,
            'hotel_name': hotel_name,
            'page_number': page_number,
            'title': title
        },
        'content': content
    }

    records.append(doc)

print(records[0])

{'metadata': {'id': 'grand-azure-bay-hotel-1', 'hotel_name': 'Grand Azure Bay Hotel', 'page_number': 1, 'title': 'Overview'}, 'content': 'Overview : Grand Azure Bay Hotel is a luxury beachfront property offering premium accommodation, fine dining, and personalized guest experiences. The hotel is designed to cater to both business and leisure travelers with a focus on comfort, safety, and service excellence. : '}


In [7]:
from unstructured.staging.base import elements_from_base64_gzipped_json

records = []

# hotel name from the document title (first chunk), split on " - " 
doc_title = str(chunks[0]).strip()
hotel_name = doc_title.split(" - ")[0].strip()          # "Grand Azure Bay Hotel"
hotel_slug = hotel_name.lower().replace(" ", "-")       # "grand-azure-bay-hotel"

for counter, element in enumerate(chunks[1:], start=1):
    metadata = element.metadata.to_dict()
    orig_elements = elements_from_base64_gzipped_json(metadata["orig_elements"])

    title = "General"
    parts = []
    for e in orig_elements:
        if e.category == "Title":
            title = e.text
        else:
            parts.append(e.text)
    narrative = " ".join(parts)

    if len(narrative) < 30:        # skip title-only / junk chunks
        continue

    records.append({
        "id": f"{hotel_slug}-{counter}",
        "content": f"{title}: {narrative}",
        "metadata": {
            "hotel_name": hotel_name,
            "section": title,
            "page_number": metadata.get("page_number", 1),
            "chunk_id": counter,
        },
    })

print(len(records))                 # expect 16
print(records[0]["content"][:100])  # expect "Overview: Grand Azure Bay Hotel is a luxury..."

15
Overview: Grand Azure Bay Hotel is a luxury beachfront property offering premium accommodation, fine


In [14]:
records

[{'id': 'grand-azure-bay-hotel-1',
  'content': 'Overview: Grand Azure Bay Hotel is a luxury beachfront property offering premium accommodation, fine dining, and personalized guest experiences. The hotel is designed to cater to both business and leisure travelers with a focus on comfort, safety, and service excellence.',
  'metadata': {'hotel_name': 'Grand Azure Bay Hotel',
   'section': 'Overview',
   'page_number': 1,
   'chunk_id': 1}},
 {'id': 'grand-azure-bay-hotel-2',
  'content': 'Location & Accessibility: The hotel is located in Elysian Coast City, approximately 5 km from the city center and 20 km from the international airport. It is easily accessible via taxi, public transport, and hotel-arranged shuttle services. Nearby attractions include beaches, cultural landmarks, shopping districts, and nightlife hubs.',
  'metadata': {'hotel_name': 'Grand Azure Bay Hotel',
   'section': 'Location & Accessibility',
   'page_number': 1,
   'chunk_id': 2}},
 {'id': 'grand-azure-bay-hotel-

# create embeddings and push to db

In [8]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

client = chromadb.PersistentClient(path="./chroma_db")
ef = SentenceTransformerEmbeddingFunction(model_name="BAAI/bge-small-en-v1.5")
collection = client.get_or_create_collection(
    name="hotel_docs",
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"},
)

collection.upsert(
    ids=[r["id"] for r in records],
    documents=[r["content"] for r in records],
    metadatas=[r["metadata"] for r in records],
)
print(f"Loaded {collection.count()} chunks")

# ---- the four-query retrieval test ----
for q in [
    "What is the famous dish in the hotel?",
    "How does the hotel ensure hygiene?",
    "Is vegetarian food available?",
    "What is the cancellation policy?",
]:
    res = collection.query(query_texts=[q], n_results=3)
    print(q, "->", [m["section"] for m in res["metadatas"][0]])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded 15 chunks
What is the famous dish in the hotel? -> ['Dining Experience', 'Famous Dishes', 'Common User Questions - Food']
How does the hotel ensure hygiene? -> ['Hygiene & Cleanliness Protocols', 'Common User Questions - Safety', 'Common User Questions - Food']
Is vegetarian food available? -> ['Common User Questions - Food', 'Famous Dishes', 'Common User Questions - Safety']
What is the cancellation policy? -> ['Cancellation & Modification Policy', 'Common User Questions - Booking', 'Reservation Process']


In [9]:
collection.query(query_texts="How does the hotel ensure hygiene?", n_results=3)

{'ids': [['grand-azure-bay-hotel-4',
   'grand-azure-bay-hotel-7',
   'grand-azure-bay-hotel-11']],
 'embeddings': None,
 'documents': [['Hygiene & Cleanliness Protocols: The hotel follows strict hygiene standards aligned with international hospitality guidelines. Rooms are deep-cleaned after every checkout using hospital-grade disinfectants. High-touch areas such as door handles, elevators, and reception counters are sanitized multiple times a day.',
   'Common User Questions - Safety: Guests often ask whether the hotel follows COVID or health safety guidelines, whether food is hygienic, and how frequently rooms are cleaned. The hotel ensures full transparency and compliance with safety protocols.',
   'Common User Questions - Food: Guests frequently ask about vegetarian options, allergen information, spice levels, and availability of customized meals. The hotel accommodates dietary restrictions and provides detailed menu information.']],
 'uris': None,
 'included': ['metadatas', 'doc

# cross - encoder reranking

In [ ]:
from sentence_transformers import CrossEncoder

rerank_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

In [ ]:
"""Evaluated a cross-encoder re-ranker (mxbai-rerank-xsmall); scores confirmed sharper discrimination, 
but at 15 chunks top-3 recall was already 100%, so it's excluded from the pipeline to keep latency and dependencies minimal."""

from sentence_transformers import CrossEncoder

# Load the model, here we use our base sized model
model = CrossEncoder("mixedbread-ai/mxbai-rerank-xsmall-v1")

# Example query and documents
query = "How does the hotel ensure hygiene?"
documents = ['Hygiene & Cleanliness Protocols: The hotel follows strict hygiene standards aligned with international hospitality guidelines. Rooms are deep-cleaned after every checkout using hospital-grade disinfectants. High-touch areas such as door handles, elevators, and reception counters are sanitized multiple times a day.',
   'Common User Questions - Safety: Guests often ask whether the hotel follows COVID or health safety guidelines, whether food is hygienic, and how frequently rooms are cleaned. The hotel ensures full transparency and compliance with safety protocols.',
   'Common User Questions - Food: Guests frequently ask about vegetarian options, allergen information, spice levels, and availability of customized meals. The hotel accommodates dietary restrictions and provides detailed menu information.']
# Lets get the scores
results = model.rank(query, documents, return_documents=True, top_k=3)

results

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

# database connection

In [14]:
"""SQLite connection + schema for reservations."""

import sqlite3

import sys
from pathlib import Path

# Find the project root (one level up from 'experiments') and add it to sys.path
PROJECT_ROOT = str(Path('.').resolve().parent)
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)


from src.config import DB_PATH

SCHEMA = """
CREATE TABLE IF NOT EXISTS reservations (
    reservation_id  TEXT PRIMARY KEY,
    hotel_id        TEXT NOT NULL,
    guest_name      TEXT NOT NULL,
    email           TEXT NOT NULL,
    check_in        TEXT NOT NULL,   -- ISO YYYY-MM-DD (SQLite has no native date type)
    check_out       TEXT NOT NULL,
    room_type       TEXT NOT NULL DEFAULT 'standard',
    status          TEXT NOT NULL DEFAULT 'active'
                    CHECK (status IN ('active', 'cancelled')),
    created_at      TEXT NOT NULL,
    CHECK (check_out > check_in)
);
CREATE INDEX IF NOT EXISTS idx_res_email ON reservations(email);
"""

def get_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row  # rows behave like dicts: row["email"]
    conn.executescript(SCHEMA)
    return conn

# Reservation Tool

In [12]:
import re
import secrets
from datetime import datetime, date

In [15]:
def valid_date(value):
    try:
        return datetime.strptime(value, "%Y-%m-%d").date()
    except (ValueError, TypeError):
        return None

In [16]:
def create_reservation(guest_name, email, check_in, check_out,
                       room_type="standard"):
    if not guest_name or not guest_name.strip():
        return {"ok": False, "error": "Guest name is required."}
    if not re.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", email or ""):
        return {"ok": False, "error": "A valid email address is required."}

    ci, co = valid_date(check_in), valid_date(check_out)
    if not ci or not co:
        return {"ok": False, "error": "Dates must be in YYYY-MM-DD format."}
    if ci < date.today():
        return {"ok": False, "error": "Check-in date cannot be in the past."}
    if co <= ci:
        return {"ok": False, "error": "Check-out must be after check-in."}
    if room_type not in {"standard", "deluxe", "suite"}:
        return {"ok": False, "error": f"Room type must be one of: {', '.join(sorted({"standard", "deluxe", "suite"}))}."}

    rid = f"RES-{secrets.token_hex(3).upper()}"
    conn = get_connection()
    try:
        conn.execute(
            "INSERT INTO reservations VALUES (?,?,?,?,?,?,?,?,?)",
            (rid, "grand-azure-bay-hotel", guest_name.strip(), email.strip().lower(),
             check_in, check_out, room_type, "active",
             datetime.now().isoformat(timespec="seconds")),
        )
        conn.commit()
    finally:
        conn.close()

    return {"ok": True, "reservation_id": rid, "check_in": check_in,
            "check_out": check_out, "room_type": room_type}

In [17]:
NOT_FOUND = {"ok": False, "error": "No reservation found with that ID and email."}

def get_reservation(reservation_id, email):
    conn = get_connection()
    try:
        row = conn.execute(
            "SELECT * FROM reservations WHERE reservation_id = ? AND email = ?",
            (reservation_id.strip().upper(), email.strip().lower()),
        ).fetchone()
    finally:
        conn.close()

    if row is None:
        return NOT_FOUND
    return {"ok": True, "reservation_id": row["reservation_id"],
            "guest_name": row["guest_name"], "check_in": row["check_in"],
            "check_out": row["check_out"], "room_type": row["room_type"],
            "status": row["status"]}

In [18]:
def cancel_reservation(reservation_id, email):
    conn = get_connection()
    try:
        row = conn.execute(
            "SELECT status FROM reservations WHERE reservation_id = ? AND email = ?",
            (reservation_id.strip().upper(), email.strip().lower()),
        ).fetchone()
        if row is None:
            return NOT_FOUND
        if row["status"] == "cancelled":
            return {"ok": True, "reservation_id": reservation_id,
                    "status": "cancelled", "note": "This reservation was already cancelled."}
        conn.execute(
            "UPDATE reservations SET status = 'cancelled' WHERE reservation_id = ?",
            (reservation_id.strip().upper(),),
        )
        conn.commit()
    finally:
        conn.close()

    return {"ok": True, "reservation_id": reservation_id, "status": "cancelled"}

In [19]:
r = create_reservation("Test Guest", "Test@Example.com", "2026-07-25", "2026-07-27")
print(r)
print(create_reservation("karthick", "bad-email", "2026-07-25", "2026-07-27"))
print(create_reservation("suriya", "suriya@b.com", "2020-01-01", "2020-01-05"))
print(get_reservation(r["reservation_id"], "test@example.com"))     # owner -> full record
print(get_reservation(r["reservation_id"], "hacker@evil.com"))      # wrong email -> NOT_FOUND
print(get_reservation("RES-000000", "test@example.com"))            # wrong id -> same message
print(cancel_reservation(r["reservation_id"], "test@example.com"))
print(cancel_reservation(r["reservation_id"], "test@example.com"))  # already cancelled

{'ok': True, 'reservation_id': 'RES-E56C54', 'check_in': '2026-07-25', 'check_out': '2026-07-27', 'room_type': 'standard'}
{'ok': False, 'error': 'A valid email address is required.'}
{'ok': False, 'error': 'Check-in date cannot be in the past.'}
{'ok': True, 'reservation_id': 'RES-E56C54', 'guest_name': 'Test Guest', 'check_in': '2026-07-25', 'check_out': '2026-07-27', 'room_type': 'standard', 'status': 'active'}
{'ok': False, 'error': 'No reservation found with that ID and email.'}
{'ok': False, 'error': 'No reservation found with that ID and email.'}
{'ok': True, 'reservation_id': 'RES-E56C54', 'status': 'cancelled'}
{'ok': True, 'reservation_id': 'RES-E56C54', 'status': 'cancelled', 'note': 'This reservation was already cancelled.'}


In [ ]:
conn = get_connection()

query = 'SELECT * FROM reservations'

row = conn.execute(query).fetchall()

for r in row:
    print(r.keys())

['reservation_id', 'hotel_id', 'guest_name', 'email', 'check_in', 'check_out', 'room_type', 'status', 'created_at']
['reservation_id', 'hotel_id', 'guest_name', 'email', 'check_in', 'check_out', 'room_type', 'status', 'created_at']


# search hotelm info 